In [47]:
import os
import pprint
import tempfile

from typing import Dict, Text
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import tensorflow_recommenders as tfrs

In [ ]:
# import warnings
# warnings.filterwarnings('ignore')

데이터 전처리 과정

1. 데이터 불러오기
2. 필요한 col만 남기고 지우기 (유저아이디-영화제목/ 영화제목)
3. Train/ Test 분할
3. unique 변수 생성

In [48]:
# --- 데이터 세트 준비
ratings = tfds.load("movielens/100k-ratings", split="train")
movies = tfds.load("movielens/100k-movies", split="train")

In [49]:
for x in ratings.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'bucketized_user_age': 45.0,
 'movie_genres': array([7]),
 'movie_id': b'357',
 'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'raw_user_age': 46.0,
 'timestamp': 879024327,
 'user_gender': True,
 'user_id': b'138',
 'user_occupation_label': 4,
 'user_occupation_text': b'doctor',
 'user_rating': 4.0,
 'user_zip_code': b'53211'}


2026-03-07 23:25:44.028034: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [50]:
for x in movies.take(1).as_numpy_iterator():
    pprint.pprint(x)

{'movie_genres': array([4]),
 'movie_id': b'1681',
 'movie_title': b'You So Crazy (1994)'}


2026-03-07 23:25:44.068093: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [51]:
# --- 영화제목, 유저아이디 빼고 지우기
ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
})
movies = movies.map(lambda x: x["movie_title"])

In [ ]:
for x in ratings.take(3).as_numpy_iterator():
    pprint.pprint(x)

print(len(ratings)) # 100,000개 

{'movie_title': b"One Flew Over the Cuckoo's Nest (1975)", 'user_id': b'138'}
{'movie_title': b'Strictly Ballroom (1992)', 'user_id': b'92'}
{'movie_title': b'Very Brady Sequel, A (1996)', 'user_id': b'301'}
100000


2026-03-07 23:25:44.151618: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [ ]:
for x in movies.take(3).as_numpy_iterator():
    pprint.pprint(x)
    
print(len(movies)) # 1682개

b'You So Crazy (1994)'
b'Love Is All There Is (1996)'
b'Fly Away Home (1996)'
1682


2026-03-07 23:25:44.199116: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [54]:
# --- Train/ Test 분할 (랭킹데이터만) 
# (데이터 편향 고치려고 셔플.(멜로영화만나오다액션영화쭉나오면 .. 학습이 제대로 안 되겠지?))
tf.random.set_seed(42)

shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)
# reshuffle_each_iteration 이거 끈 이유는 학습할때 본 문제를 테스트때 보면 안 되어서 (데이터 누수)

train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [55]:
# unique 변수 생성 (vocabulary로 쓸 거임.)
movie_titles = movies.batch(1_000) # movies에 영화제목만 있음
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"]) # 랭킹에 유저아이디, 그에맞는 영화 있는거 유저아이디만 맵핑

unique_movie_titles = np.unique(np.concatenate(list(movie_titles))) # 리스트타입이 아님 왜냐면 위에서 배치 설정을 했잖슴! 그럼 데이터셋이겟지(흐르는 강물)
unique_user_ids = np.unique(np.concatenate(list(user_ids))) # np.unique는 중복제거, 정렬해주는 정리도구

unique_movie_titles[:10]

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)',
       b'2 Days in the Valley (1996)',
       b'20,000 Leagues Under the Sea (1954)',
       b'2001: A Space Odyssey (1968)',
       b'3 Ninjas: High Noon At Mega Mountain (1998)',
       b'39 Steps, The (1935)'], dtype=object)

1. **movie_titles**: 데이터 셋임 [1000개 뭉치]->[1000개 뭉치]->[1000개 뭉치] (리스트가) 흐르는 중
2. **list(movie_titles)**: 리스트에 담김 [[1000개],[1000개],[1000개]] 바구니에 담김 흐르지않음. 
                        
                        근데 리스트 안에 리스트?!(이중 리스트)
3. **np.concatenate(list(movie_titles))**: [ 1000개+1000개+1000개 ] 하나의 리스트가됨.


리스트는 멈춰서 쌓인 상태

데이터셋은 흐르는 상태

In [56]:
# --- 모델 구현
embedding_dimension = 32

# 쿼리 타워
user_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_user_ids, mask_token=None),
    tf.keras.layers.Embedding(len(unique_user_ids)+1, embedding_dimension)
])

# 후보 타워
movie_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_movie_titles, mask_token=None),
    tf.keras.layers.Embedding(len(unique_movie_titles)+1, embedding_dimension)
])

In [57]:
# Train에 긍정적인 (사용자, 영화)쌍이 있다.
# 모델이 얼마나 좋은지 알아내려면
# 다른 쌍들보다 긍정적인 쌍이 훨씬 점수가 높으면 된다.

# --- 채점표 (재료)
metrics = tfrs.metrics.FactorizedTopK( # 채점표 안에 후보자들을 넣는데.
    candidates=movies.batch(128).map(movie_model) # 후보자들이란: 32개로 쪼갠 영화이름 데이터들을 영화 모델에 대응시킨 것.
    # 후보자들 = 영화리스트(객관식 보기)
)

# --- 채점기 (완성)
# 얼마나 정답에 가까운지 점수 매기고 (Retrieval: 검색하고)
# Loss:벌점 줌.
task = tfrs.tasks.Retrieval(
    metrics=metrics
)

ARE YA WINNING SON?!!!!!
암오케
<img src="https://i.pinimg.com/736x/58/4a/30/584a309a747e8925bde7810979ece29c.jpg" width="50%">

# --- 전체 모델
class MovielensModel(tfrs.Model): # tfrs.Model 기본모델이 부모라고 선언
    
    def __init__(self, user_model, movie_model): 
        super().__init__() # 부모님의.기본기능들 세팅
        self.movie_model: tf.keras.Model = movie_model
        self.user_model: tf.keras.Model = user_model
        self.task: tf.keras.layers.Layer = task
        
    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
        user_embeddings = self.user_model(features["user_id"])
        positive_movie_embeddings = self.movie_model(features["movie_title"])
        return self.task(user_embeddings, positive_movie_embeddings)

In [ ]:
class NoBaseClassMovielensModel (tf.keras.Model):
    def __init__(self, user_model, movie_model):
        super().__init__()
        self.user_model: tf.keras.Model = user_model
        self.movie_model: tf.keras.Model = movie_model
        self.task: tf.keras.layers.Layer = task
        
    def train_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        with tf.GradientTape() as tape: # GradientTape은 미분을 자동으로 계산. (기울기 계산용임.)
            
            user_embeddings = self.user_model(features["user_id"])
            positive_movie_embeddings = self.movie_model(features["movie_title"])
            
            loss = self.task(
                user_embeddings, 
                positive_movie_embeddings,
                # compute_metrics=False 전수 대조 끄기
            )
            regularization_loss = sum(self.losses)
            total_loss = loss + regularization_loss
            
        gradients = tape.gradient(total_loss, self.trainable_variables) #모든 손실에 대한 "학습가능한(업데이트가능한) 가준치와 편향 변수들의 목록"-> 미분값을 계산
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables)) # 위에서 계산한 미분값을 1:1로 변수들에게 적용
        
        metrics = {metric.name: metric.result() for metric in self.metrics} # 이름은 보통 factorized_top_k << 이게 나옴
        metrics["loss"] = loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        
        return metrics
        
    # train_step에서는 가중치 업데이트를 하지만 test에서는 현상유지
    def test_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        user_embeddings = self.user_model(features["user_id"])
        positive_movie_embeddings = self.movie_model(features["movie_title"])
        
        loss = self.task(user_embeddings, positive_movie_embeddings)
        regularization_loss = sum(self.losses)
        total_loss = loss + regularization_loss
        
        metrics = {metric.name: metric.result() for metric in self.metrics} # 예쁜 딕셔너리 형태로.. 보기위해서.
        metrics["loss"] = loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        
        return metrics

In [61]:
# 모델 정의 및 컴파일
model = NoBaseClassMovielensModel(user_model, movie_model)
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

In [63]:
# gpu나 메모리가 데이터를 주고받을 때 딱 떨어지는 2진수 batch로 주면 속도가 빠르다
cached_train = train.shuffle(100_000).batch(8192).cache()
cached_test = test.batch(4096).cache()
# train만 섞는 이유는 에포크(복습) 때문
# 1에포크 끝나고 다음 2에포크때 외워서 맞추면 안 돼서.
# 십만 개 셔플한 거는 걍 넉넉하게

In [64]:
model.fit(cached_train, epochs=3)

Epoch 1/3


2026-03-11 12:48:15.769634: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


10/10 [==============================] - 14s 578ms/step - factorized_top_k/top_1_categorical_accuracy: 5.5000e-04 - factorized_top_k/top_5_categorical_accuracy: 0.0053 - factorized_top_k/top_10_categorical_accuracy: 0.0121 - factorized_top_k/top_50_categorical_accuracy: 0.0851 - factorized_top_k/top_100_categorical_accuracy: 0.1635 - loss: 69829.3423 - regularization_loss: 0.0000e+00 - total_loss: 69829.3423
Epoch 2/3
10/10 [==============================] - 5s 509ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0019 - factorized_top_k/top_5_categorical_accuracy: 0.0161 - factorized_top_k/top_10_categorical_accuracy: 0.0346 - factorized_top_k/top_50_categorical_accuracy: 0.1582 - factorized_top_k/top_100_categorical_accuracy: 0.2835 - loss: 67463.1406 - regularization_loss: 0.0000e+00 - total_loss: 67463.1406
Epoch 3/3
10/10 [==============================] - 5s 505ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0024 - factorized_top_k/top_5_categorical_accuracy: 0.021

---

모델이 학습됨에 따라 손실이 감소되고,

top-k 검색 메트릭 세트가 업데이트됩니다.

이는 전체 후보 집합에서 검색된 상위 k항목에 긍정영화목록이 있는지 여부를 알려줍니다.

예를들어, 상위 5개 범주 메트릭이 0.2면,

평균적으로 긍정영화목록이 5개 항목의 20%에 해당.

이 예에서는 평가뿐만 아니라 훈련 중에도 메트릭을 평가합니다.

대규모 후보집합에서는 이 작업이 매우 느릴 수 있으므로,

 Train에서는 메트릭 계산을 끄고 평가에서만 실행하는 것이 현명할 수 있습니다.

 > 최소 대조(loss), 전수 대조(metric) 구분을 잘 하자.
 
 > 최소대조-batch안에서만 잘하면되지.
 전수대조-FactorizedTopK (전교생과 비교)

In [65]:
model.evaluate(cached_test, return_dict=True)

5/5 [==============================] - 3s 441ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0010 - factorized_top_k/top_5_categorical_accuracy: 0.0093 - factorized_top_k/top_10_categorical_accuracy: 0.0213 - factorized_top_k/top_50_categorical_accuracy: 0.1218 - factorized_top_k/top_100_categorical_accuracy: 0.2333 - loss: 31077.5459 - regularization_loss: 0.0000e+00 - total_loss: 31077.5459


{'factorized_top_k/top_1_categorical_accuracy': 0.0010499999625608325,
 'factorized_top_k/top_5_categorical_accuracy': 0.009349999949336052,
 'factorized_top_k/top_10_categorical_accuracy': 0.021299999207258224,
 'factorized_top_k/top_50_categorical_accuracy': 0.121799997985363,
 'factorized_top_k/top_100_categorical_accuracy': 0.23330000042915344,
 'loss': 28258.21484375,
 'regularization_loss': 0,
 'total_loss': 28258.21484375}

In [ ]:
# 예측하기
index = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
index.index_from_dataset(
    tf.data.Dataset.zip((movies.batch(100), movies.batch(100).map(model.movie_model)))
)

_, titles = index(tf.constant(["42"]))
print(f"추천: {titles[0, :3]}")

추천: [b'Rudy (1993)' b'Rent-a-Kid (1995)' b'Aristocats, The (1970)']


- 내가 index(유저아이디)이렇게 입력하면
- user_model이 32개 임베딩숫자로 바꾸어줌.
- 다음엔 영화창고로 달려가 수만개의 영화임베딩숫자들을 일일히 대조함(내적(Dot Product)이라는곱셈계산을한다.)
- ((영화창고에 이미 영화임베딩숫자들로 저장해둔상태))

- 대조가 끝나면 점수가 제일 높은 순서대로 줄을 세움(Top-K정렬)

In [73]:
# 모델 서빙하기 (배포테스트)
with tempfile.TemporaryDirectory() as tmp: # 1회용 가상 폴더
    path = os.path.join(tmp, "model")
    tf.saved_model.save(index, path)
    loaded = tf.saved_model.load(path) # 파일로 저장되었다가 다시 불려와도 가중치가 파괴되지않고 잘 작동되어야한다.
    
    scores, titles = loaded(["42"])
    print(f"추천: {titles[0, :3]}")

INFO:tensorflow:Assets written to: /var/folders/c5/cl6djkjd0b3g6dptld67qj4r0000gn/T/tmp03qof6qn/model/assets


INFO:tensorflow:Assets written to: /var/folders/c5/cl6djkjd0b3g6dptld67qj4r0000gn/T/tmp03qof6qn/model/assets


추천: [b'Rudy (1993)' b'Rent-a-Kid (1995)' b'Aristocats, The (1970)']
